# core

> Fill in a module description here

In [1]:
# | default_exp extraction

In [1]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [152]:
# | export
from enum import Enum
from typing import List, Union, Optional
from pydantic import BaseModel, Field
from instructor import patch, Mode
from litellm import LiteLLM
import litellm
from dotenv import load_dotenv

litellm.drop_params = True

load_dotenv()

True

In [180]:
# | export

class ModelType(Enum):
    CLAUDE_3_HAIKU = "claude-3-haiku-20240307"
    CLAUDE_3_SONNET = "claude-3-sonnet-20240229"
    GPT_4_0125_PREVIEW = "gpt-4-0125-preview"
    GPT_3_5_TURBO = "gpt-3.5-turbo"
    CLAUDE_3_OPUS = "claude-3-opus-20240229"

class QuestionSchema(BaseModel):
    id: str
    text: str
    output_type: str
    categories: Optional[List[str]] = None

class Questions(BaseModel):
    questions: List[QuestionSchema]

class AnswerSchema(BaseModel):
    question_id: str
    text: str = Field(..., title="The answer to the question")

class AnswerCategorySchema(BaseModel):
    question_id: str
    categories: Union[str, List[str]] = Field(..., title="The answer to the question. Must be a valid category or list of category from the originally provided text.")

class Answers(BaseModel):
    answers: List[Union[AnswerSchema, AnswerCategorySchema]]

class AIModelClient:
    """
    A client for interacting with various language models to create schemas and extract information.

    Args:
        model_type (ModelType): The type of language model to use.
    """
    def __init__(self, model_type: ModelType=ModelType.CLAUDE_3_HAIKU, max_retries: int = 10, max_tokens: int = 4096):
        self.client = patch(LiteLLM(), mode=Mode.JSON) # type: ignore
        self.model_type = model_type
        self.max_tokens = max_tokens
        self.max_retries = max_retries
        self.start_with = "{"


    def create_schema(self, questions: str, model_type_override: Optional[ModelType] = None) -> Questions:
        """
        Create a schema from the given questions.

        Args:
            questions (str): The questions to be structured.

        Returns:
            Questions: The structured questions.
        """
        response: Questions = self.client.chat.completions.create( # type: ignore
            model=model_type_override.value if model_type_override else self.model_type.value,
            response_model=Questions,
            max_retries=self.max_retries,
            max_tokens=self.max_tokens,
            messages=[
                {
                    "role": "system",
                    "content": f"Extract a set of structured questions from the user-provided unstructured text using ONLY JSON with NO preamble so it can be validated immediately, with no spacing or newlines, following the following JSON schema format: {Questions.model_json_schema()}. i.e., start your response with {self.start_with}. You should respond in a message, not a tool or function call, if that's an option for you. Output type can be 'string', or 'category'. If output type is 'category', then the categories key should be a list of relevant categories provided by the user."
                },
                {
                    "role": "user",
                    "content": questions,
                },
            ],
        )
        if isinstance(response, Questions):
            return response
        else:
            raise ValueError("Response is not of type Questions")

    def extract_information(self, text: str, schema: Questions, model_type_override: Optional[ModelType] = ModelType.CLAUDE_3_OPUS) -> Answers:
        """
        Extract information from the given text using the provided schema.

        Args:
            text (str): The text to extract information from.
            schema (Questions): The schema to use for extraction.

        Returns:
            Answers: The extracted information.
        """
        response: Answers = self.client.chat.completions.create( # type: ignore
            model=model_type_override.value if model_type_override else self.model_type.value,
            response_model=Answers,
            max_retries=self.max_retries,
            max_tokens=self.max_tokens,
            messages=[
                {
                    "role": "system",
                    "content": f"Answer the following questions based on the user-provided text using ONLY JSON with NO preamble so it can be validated immediately, with no spacing or newlines, following the following JSON schema format: {Answers.model_json_schema()}, i.e., start your response with {self.start_with}. You should respond in a message, not a tool or function call, if that's an option for you. Be concise, and answer only the question or questions provided: {schema.model_dump_json()}. Answer these questions in the most intelligent way possible, as this is high-stakes for the user to get it right.",
                },
                {"role": "user", "content": text},
            ],
        )
        if isinstance(response, Answers):
            return response
        else:
            raise ValueError("Response is not of type Answers")


In [181]:
EXAMPLE_TEXT = """Speaker A: Smoke from hundreds of wildfires in Canada is triggering air quality alerts throughout the US. Skylines from Maine to Maryland to Minnesota are gray and smoggy. And in some places, the air quality warnings include the warning to stay inside. We wanted to better understand what's happening here and why, so we called Peter DiCarlo, an associate professor in the department of Environmental Health and Engineering at Johns Hopkins University. Good morning. Professor Good morning. So what is it about the conditions right now that have caused this round of wildfires to affect so many people so far away?
Speaker B: Well, there's a couple of things. The season has been pretty dry already, and then the fact that we're getting hit in the US is because there's a couple of weather systems that are essentially channeling the smoke from those Canadian wildfires through Pennsylvania into the mid Atlantic and the northeast and kind of just dropping the smoke there.
Speaker A: So what is it in this haze that makes it harmful? And I'm assuming it is.
Speaker B: Is it is. The levels outside right now in Baltimore are considered unhealthy. And most of that is due to what's called particulate matter, which are tiny particles, microscopic, smaller than the width of your hair, that can get into your lungs and impact your respiratory system, your cardiovascular system, and even your neurological, your brain.
Speaker A: What makes this particularly harmful? Is it the volume of particulate? Is it something in particular? What is it exactly? Can you just drill down on that a little bit more? Yeah.
Speaker B: So the concentration of particulate matter I was looking at, some of the monitors that we have was reaching levels of what are, in science speak, 150 micrograms per meter cubed, which is more than ten times what the annual average should be and about four times higher than what you're supposed to have on a 24 hours average. And so the concentrations of these particles in the air are just much, much higher than we typically see. And exposure to those high levels can lead to a host of health problems.
Speaker A: And who is most vulnerable? I noticed that in New York City, for example, they're canceling outdoor activities. And so here it is in the early days of summer, and they have to keep all the kids inside. So who tends to be vulnerable in a situation like this?
Speaker B: It's the youngest. So children, obviously, whose bodies are still developing, the elderly who know their bodies are more in decline and they're more susceptible to the health impacts of breathing, the poor air quality. And then people who have preexisting health conditions, people with respiratory conditions or heart conditions, can be triggered by high levels of air pollution.
Speaker A: Could this get worse?
Speaker B: That's a good, in some areas it's much worse than others, and it just depends on kind of where the smoke is concentrated. I think New York has some of the higher concentrations right now, but that's going to change as that air moves away from the New York area. But over the course of the next few days, we will see different areas being hit at different times with the highest concentrations. I was going to ask you more fires start burning. I don't expect the concentrations to go up too much higher.
Speaker A: I was going to ask you, and you started to answer this, but how much longer could this last? Or, forgive me if I'm asking you to speculate, but what do you think?
Speaker B: Well, I think the fires are going to burn for a little bit longer, but the key for us in the US is the weather system changing. And so right now it's kind of the weather systems that are pulling that air into our mid atlantic and northeast region. As those weather systems change and shift, we'll see that smoke going elsewhere and not impact us in this region as much. And so I think that's going to be the defining factor. And I think the next couple of days we're going to see a shift in that weather pattern and start to push the smoke away from where we are.
Speaker A: And finally, with the impacts of climate change, we are seeing more wildfires. Will we be seeing more of these kinds of wide ranging air quality consequences or circumstances?
Speaker B: I mean, that is one of the predictions for climate change. Looking into the future, the fire season is starting earlier and lasting longer, and we're seeing more frequent fires. So, yeah, this is probably something that we'll be seeing more frequently. This tends to be much more of an issue in the western Us. So the eastern us getting hit right now is a little bit new. But, yeah, I think with climate change moving forward, this is something that is going to happen more frequently.
Speaker A: That's Peter DiCarlo, associate professor in the department of Environmental Health and engineering at Johns Hopkins University. Thanks so much for joining us and sharing this expertise with us.
Speaker B: Thank you for having me."""

questions = """1. What is the name of speaker A, if mentioned?
2. What is the name of speaker B, if mentioned?
3. What are they talking about?
4. Pick from the following list of tags: smoke, air quality, health, climate change, and pick as many as are relevant that apply to the conversation."""

def test_model_client_create():
    ai_client = AIModelClient()

    # Test create_schema
    schema = ai_client.create_schema(questions)

    assert isinstance(schema, Questions)
    assert len(schema.questions) == 4
    assert schema.questions[0].id == '1'
    assert 'What is the name of speaker A' in schema.questions[0].text
    assert schema.questions[0].output_type == 'string'
    assert len(schema.questions[3].categories) == 4
    return schema

def test_model_client_extract(schema):
    """Going to be a bit flaky if using haiku"""
    ai_client = AIModelClient(ModelType.CLAUDE_3_HAIKU)
    # Test extract_information
    answers = ai_client.extract_information(EXAMPLE_TEXT, schema)

    assert isinstance(answers, Answers)
    assert len(answers.answers) == 4
    assert answers.answers[0].question_id == '1'
    assert len(answers.answers[0].text) > 0
    assert len(answers.answers[3].categories) > 0

schema = test_model_client_create()
test_model_client_extract(schema)


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore